In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import math
import os

import numpy as np
import torch
from torch_geometric import seed_everything

seed_everything(42)

cache_dir = "./.cache_predql"

/home/kolesiko/CTU/BT/predql-tasks/.venv/lib/python3.12/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: /home/kolesiko/CTU/BT/predql-tasks/.venv/lib/python3.12/site-packages/torch_scatter/_version_cuda.so: undefined symbol: _ZN3c106detail14torchCheckFailEPKcS2_jRKNSt7__cxx1112basic_stringIcSt11char_traitsIcESaIcEEE
  import torch_geometric.typing
/home/kolesiko/CTU/BT/predql-tasks/.venv/lib/python3.12/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-cluster'. Disabling its usage. Stacktrace: /home/kolesiko/CTU/BT/predql-tasks/.venv/lib/python3.12/site-packages/torch_cluster/_version_cuda.so: undefined symbol: _ZN3c106detail14torchCheckFailEPKcS2_jRKNSt7__cxx1112basic_stringIcSt11char_traitsIcESaIcEEE
  import torch_geometric.typing
/home/kolesiko/CTU/BT/predql-tasks/.venv/lib/python3.12/site-packages/torch_geometric/__init__.py:4: UserWarning: 

In [3]:
from experiments.model_training.utils import get_device

device = get_device()

Using device: cuda


In [4]:
import redelex

from relbench.datasets import get_dataset
from relbench.modeling.utils import get_stype_proposal
from predql_tasks.tasks import SFScoresBusinessesScoresTmpTask

In [5]:
dataset_name = "ctu-sfscores"
task_name = "sfscores_businesses_scores_tmp"

dataset = get_dataset("ctu-sfscores", download=False)
task = SFScoresBusinessesScoresTmpTask()
db = dataset.get_db()

Loading Database object from /home/kolesiko/.cache/relbench/ctu-sfscores/db...
Done in 0.03 seconds.


In [6]:
col_to_stype_dict = get_stype_proposal(db)
col_to_stype_dict

{'violations': {'__PK__': <stype.numerical: 'numerical'>,
  'business_id': <stype.numerical: 'numerical'>,
  'date': <stype.timestamp: 'timestamp'>,
  'violation_type_id': <stype.timestamp: 'timestamp'>,
  'risk_category': <stype.categorical: 'categorical'>,
  'description': <stype.text_embedded: 'text_embedded'>,
  'FK_businesses_business_id': <stype.numerical: 'numerical'>},
 'businesses': {'__PK__': <stype.numerical: 'numerical'>,
  'name': <stype.text_embedded: 'text_embedded'>,
  'address': <stype.text_embedded: 'text_embedded'>,
  'city': <stype.text_embedded: 'text_embedded'>,
  'postal_code': <stype.text_embedded: 'text_embedded'>,
  'latitude': <stype.numerical: 'numerical'>,
  'longitude': <stype.numerical: 'numerical'>,
  'phone_number': <stype.numerical: 'numerical'>,
  'tax_code': <stype.text_embedded: 'text_embedded'>,
  'business_certificate': <stype.numerical: 'numerical'>,
  'application_date': <stype.timestamp: 'timestamp'>,
  'owner_name': <stype.text_embedded: 'text

In [7]:
from experiments.model_training.training.text_embedder import TextEmbedder 
from experiments.model_training.utils import patched_to_unix_time

import relbench.modeling.graph
import relbench.modeling.utils
from relbench.modeling.graph import make_pkey_fkey_graph
from torch_frame.config.text_embedder import TextEmbedderConfig


text_embedder = TextEmbedderConfig(
    text_embedder=TextEmbedder(
        "BAAI/bge-base-en-v1.5",
        device=device,
        cache_dir=cache_dir), batch_size=256
)

relbench.modeling.graph.to_unix_time = patched_to_unix_time
relbench.modeling.utils.to_unix_time = patched_to_unix_time

data, col_stats_dict = make_pkey_fkey_graph(
    db, 
    col_to_stype_dict, 
    text_embedder,
    cache_dir=os.path.join(cache_dir, dataset_name)
)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/home/kolesiko/CTU/BT/predql-tasks/.venv/lib/python3.12/site-packages/torch_frame/data/dataset.py:587: UserWarning: Weights only load failed. Please file an issue to make `torch.load(weights_only=True)` compatible in your case. Please use `torch.serialization.add_safe_globals([scalar])` to allowlist this global.
  self._tensor_frame, self._col_stats = torch_frame.load(
/home/kolesiko/CTU/BT/predql-tasks/.venv/lib/python3.12/site-packages/relbench/modeling/graph.py:93: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or m

In [8]:
data

HeteroData(
  violations={
    tf=TensorFrame([31820, 5]),
    time=[31820],
  },
  businesses={ tf=TensorFrame([6358, 15]) },
  inspections={
    tf=TensorFrame([21232, 4]),
    time=[21232],
  },
  (violations, f2p_FK_businesses_business_id, businesses)={ edge_index=[2, 31820] },
  (businesses, rev_f2p_FK_businesses_business_id, violations)={ edge_index=[2, 31820] },
  (inspections, f2p_FK_businesses_business_id, businesses)={ edge_index=[2, 21232] },
  (businesses, rev_f2p_FK_businesses_business_id, inspections)={ edge_index=[2, 21232] }
)

In [9]:
from experiments.model_training.utils import make_loaders

loader_dict = make_loaders(
    data=data,
    task=task,
    batch_size=256,
    num_neighbors=[128, 64, 32],
    is_temporal=True
)

Loading Database object from /home/kolesiko/.cache/relbench/ctu-sfscores/db...
Done in 0.01 seconds.
Loading Database object from /home/kolesiko/.cache/relbench/ctu-sfscores/db...
Done in 0.01 seconds.


In [ ]:
from experiments.model_training.utils import compute_pos_weight

in_channels = 128
task_type = task.task_type
dropout = 0.4
is_temporal = True
epochs = 50

mlp_config = {
    "in_channels": 128,
    "hidden_channels": 64,
    "out_channels": 101,
    "layers": 2,
    "act": "relu",
    "norm": "batch_norm",
    "bias": True
}

## SAGE

In [15]:
from experiments.model_training.training.trainer import Trainer
from experiments.model_training.training.models.sage_model import SAGEModel
import gc


sage_config = {
    "channels": 128,
    "aggr": "mean",
    "layers": 3,
}

sage_model = SAGEModel(
    data=data,
    col_stats_dict=col_stats_dict,
    in_channels=in_channels,
    gnn_config=sage_config,
    mlp_config=mlp_config,
    task_type=task_type,
    dropout=dropout,
    is_temporal=is_temporal
).to(device)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(sage_model.parameters(), lr=3e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=5
)

trainer = Trainer(
    task=task,
    model=sage_model,
    optimizer=optimizer,
    scheduler=scheduler,
    criterion=criterion,
    device=device
)

best_weights = trainer.train(
        loader_dict,
        num_epochs=epochs,
        tune_metric="accuracy",
        higher_is_better=True,
        patience=10
)

torch.save(best_weights, os.path.join(cache_dir, dataset_name, f"{sage_model.gnn_name}_{task_name}_best_weights.pt"))
    
print(trainer.evaluate(loader_dict["train"]))
print(trainer.evaluate(loader_dict["val"]))
print(trainer.evaluate(loader_dict["test"]))

del sage_model, optimizer, scheduler, trainer
torch.cuda.empty_cache() 
gc.collect()

Training: 100%|██████████| 31/31 [00:00<00:00, 55.12it/s]


Epoch 1/50, Train Loss: 4.5355


Evaluating: 100%|██████████| 9/9 [00:00<00:00, 129.54it/s]


Epoch 1/50, Eval Metrics: {'accuracy': 0.008896797153024912, 'macro_f1': 0.0004337971347699248, 'micro_f1': 0.008896797153024912, 'loss': 4.5241064380496425}
New best model found with accuracy: 0.0089


Training: 100%|██████████| 31/31 [00:00<00:00, 52.38it/s]


...

Epoch 9/50, Train Loss: 3.0031


Evaluating: 100%|██████████| 9/9 [00:00<00:00, 130.41it/s]


Epoch 9/50, Eval Metrics: {'accuracy': 0.10720640569395018, 'macro_f1': 0.004723221197659948, 'micro_f1': 0.10720640569395018, 'loss': 4.194648079176391}
New best model found with accuracy: 0.1072


Training:  29%|██▉       | 9/31 [00:00<00:00, 47.21it/s]


KeyboardInterrupt: 

## HGT

In [17]:
from experiments.model_training.training.trainer import Trainer
from experiments.model_training.training.models.hgt_model import HGTModel
import gc


hgt_config = {
    "channels": 128,
    "heads": 4,
    "layers": 3
}

hgt_model = HGTModel(
    data=data,
    col_stats_dict=col_stats_dict,
    in_channels=in_channels,
    gnn_config=hgt_config,
    mlp_config=mlp_config,
    task_type=task_type,
    dropout=dropout,
    is_temporal=is_temporal
).to(device)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(hgt_model.parameters(), lr=3e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=5
)

trainer = Trainer(
    task=task,
    model=hgt_model,
    optimizer=optimizer,
    scheduler=scheduler,
    criterion=criterion,
    device=device
)

best_weights = trainer.train(
        loader_dict,
        num_epochs=epochs,
        tune_metric="accuracy",
        higher_is_better=True,
        patience=10
)

torch.save(best_weights, os.path.join(cache_dir, dataset_name, f"{hgt_model.gnn_name}_{task_name}_best_weights.pt"))
    
print(trainer.evaluate(loader_dict["train"]))
print(trainer.evaluate(loader_dict["val"]))
print(trainer.evaluate(loader_dict["test"]))

del hgt_model, optimizer, scheduler, trainer
torch.cuda.empty_cache() 
gc.collect()

Training: 100%|██████████| 31/31 [00:00<00:00, 41.77it/s]


Epoch 1/50, Train Loss: 4.4856


Evaluating: 100%|██████████| 9/9 [00:00<00:00, 122.25it/s]


Epoch 1/50, Eval Metrics: {'accuracy': 0.10720640569395018, 'macro_f1': 0.004723221197659948, 'micro_f1': 0.10720640569395018, 'loss': 4.602851955916109}
New best model found with accuracy: 0.1072


Training: 100%|██████████| 31/31 [00:00<00:00, 42.55it/s]


...

Evaluating: 100%|██████████| 9/9 [00:00<00:00, 134.87it/s]


{'accuracy': 0.16459074733096085, 'macro_f1': 0.026701277754248076, 'micro_f1': 0.16459074733096085, 'loss': 3.1725718983551787}


Evaluating: 100%|██████████| 7/7 [00:00<00:00, 120.38it/s]


{'accuracy': 0.15389294403892945, 'macro_f1': 0.025976820933557538, 'micro_f1': 0.15389294403892945, 'loss': 3.1781335045241383}


14436

## HAN

In [18]:
from experiments.model_training.training.trainer import Trainer
from experiments.model_training.training.models.han_model import HANModel
import gc


han_config = {
    "channels": 128,
    "heads": 2,
    "dropout": 0.7,
    "layers": 3,
}

han_model = HANModel(
    data=data,
    col_stats_dict=col_stats_dict,
    in_channels=in_channels,
    gnn_config=han_config,
    mlp_config=mlp_config,
    task_type=task_type,
    dropout=dropout,
    is_temporal=is_temporal
).to(device)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(han_model.parameters(), lr=3e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=5
)

trainer = Trainer(
    task=task,
    model=han_model,
    optimizer=optimizer,
    scheduler=scheduler,
    criterion=criterion,
    device=device
)

best_weights = trainer.train(
        loader_dict,
        num_epochs=epochs,
        tune_metric="accuracy",
        higher_is_better=True,
        patience=10
)

torch.save(best_weights, os.path.join(cache_dir, dataset_name, f"{han_model.gnn_name}_{task_name}_best_weights.pt"))
    
print(trainer.evaluate(loader_dict["train"]))
print(trainer.evaluate(loader_dict["val"]))
print(trainer.evaluate(loader_dict["test"]))

del han_model, optimizer, scheduler, trainer
torch.cuda.empty_cache() 
gc.collect()

Training: 100%|██████████| 31/31 [00:00<00:00, 45.01it/s]


Epoch 1/50, Train Loss: 4.6040


Evaluating: 100%|██████████| 9/9 [00:00<00:00, 125.49it/s]


Epoch 1/50, Eval Metrics: {'accuracy': 0.02802491103202847, 'macro_f1': 0.0013298012685881945, 'micro_f1': 0.02802491103202847, 'loss': 4.603335497642327}
New best model found with accuracy: 0.0280


Training: 100%|██████████| 31/31 [00:00<00:00, 42.55it/s]


...

Evaluating: 100%|██████████| 9/9 [00:00<00:00, 100.54it/s]


{'accuracy': 0.12544483985765126, 'macro_f1': 0.005437192711848067, 'micro_f1': 0.12544483985765126, 'loss': 3.711298550575229}


Evaluating: 100%|██████████| 7/7 [00:00<00:00, 108.12it/s]


{'accuracy': 0.11374695863746959, 'macro_f1': 0.004863332553119555, 'micro_f1': 0.11374695863746959, 'loss': 3.691551576855699}


0

## GIN

In [19]:
from experiments.model_training.training.trainer import Trainer
from experiments.model_training.training.models.gin_model import GINModel
import gc


gin_config = {
    "channels": 128,
    "mlp_layers": 2,
    "mlp_dropout": 0.3,
    "mlp_act": "relu",
    "mlp_norm": "batch_norm",
    "mlp_bias": True,
    "train_eps": True,
    "layers": 3,
}

gin_model = GINModel(
    data=data,
    col_stats_dict=col_stats_dict,
    in_channels=in_channels,
    gnn_config=gin_config,
    mlp_config=mlp_config,
    task_type=task_type,
    dropout=dropout,
    is_temporal=is_temporal
).to(device)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(gin_model.parameters(), lr=3e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=5
)

trainer = Trainer(
    task=task,
    model=gin_model,
    optimizer=optimizer,
    scheduler=scheduler,
    criterion=criterion,
    device=device
)

best_weights = trainer.train(
        loader_dict,
        num_epochs=epochs,
        tune_metric="accuracy",
        higher_is_better=True,
        patience=10
)

torch.save(best_weights, os.path.join(cache_dir, dataset_name, f"{gin_model.gnn_name}_{task_name}_best_weights.pt"))
    
print(trainer.evaluate(loader_dict["train"]))
print(trainer.evaluate(loader_dict["val"]))
print(trainer.evaluate(loader_dict["test"]))

del gin_model, optimizer, scheduler, trainer
torch.cuda.empty_cache() 
gc.collect()

Training: 100%|██████████| 31/31 [00:00<00:00, 48.76it/s]


Epoch 1/50, Train Loss: 4.5327


Evaluating: 100%|██████████| 9/9 [00:00<00:00, 107.87it/s]


Epoch 1/50, Eval Metrics: {'accuracy': 0.015569395017793594, 'macro_f1': 0.0019378298162217144, 'micro_f1': 0.015569395017793594, 'loss': 4.5411701966010805}
New best model found with accuracy: 0.0156


Training: 100%|██████████| 31/31 [00:00<00:00, 45.12it/s]


...

Evaluating: 100%|██████████| 9/9 [00:00<00:00, 109.57it/s]


{'accuracy': 0.16770462633451957, 'macro_f1': 0.02692199711269786, 'micro_f1': 0.16770462633451957, 'loss': 3.117527675798355}


Evaluating: 100%|██████████| 7/7 [00:00<00:00, 108.97it/s]


{'accuracy': 0.14659367396593673, 'macro_f1': 0.02346530812199492, 'micro_f1': 0.14659367396593673, 'loss': 3.1294326643004036}


0

## GATv2

In [20]:
from experiments.model_training.training.trainer import Trainer
from experiments.model_training.training.models.gatv2_model import GATv2Model
import gc


gatv2_config = {
    "channels": 128,
    "heads": 4,
    "add_self_loops": False,
    "layers": 3,
}

gatv2_model = GATv2Model(
    data=data,
    col_stats_dict=col_stats_dict,
    in_channels=in_channels,
    gnn_config=gatv2_config,
    mlp_config=mlp_config,
    task_type=task_type,
    dropout=dropout,
    is_temporal=is_temporal
).to(device)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(gatv2_model.parameters(), lr=3e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=5
)

trainer = Trainer(
    task=task,
    model=gatv2_model,
    optimizer=optimizer,
    scheduler=scheduler,
    criterion=criterion,
    device=device
)

best_weights = trainer.train(
        loader_dict,
        num_epochs=epochs,
        tune_metric="accuracy",
        higher_is_better=True,
        patience=10
)

torch.save(best_weights, os.path.join(cache_dir, dataset_name, f"{gatv2_model.gnn_name}_{task_name}_best_weights.pt"))
    
print(trainer.evaluate(loader_dict["train"]))
print(trainer.evaluate(loader_dict["val"]))
print(trainer.evaluate(loader_dict["test"]))

del gatv2_model, optimizer, scheduler, trainer
torch.cuda.empty_cache() 
gc.collect()

Training: 100%|██████████| 31/31 [00:00<00:00, 45.00it/s]


Epoch 1/50, Train Loss: 4.4014


Evaluating: 100%|██████████| 9/9 [00:00<00:00, 118.24it/s]


Epoch 1/50, Eval Metrics: {'accuracy': 0.12544483985765126, 'macro_f1': 0.005437192711848067, 'micro_f1': 0.12544483985765126, 'loss': 4.555599828631853}
New best model found with accuracy: 0.1254


Training: 100%|██████████| 31/31 [00:00<00:00, 44.74it/s]


...

Evaluating: 100%|██████████| 9/9 [00:00<00:00, 103.25it/s]


{'accuracy': 0.12544483985765126, 'macro_f1': 0.005437192711848067, 'micro_f1': 0.12544483985765126, 'loss': 3.993708956708263}


Evaluating: 100%|██████████| 7/7 [00:00<00:00, 95.71it/s]


{'accuracy': 0.11374695863746959, 'macro_f1': 0.004863332553119555, 'micro_f1': 0.11374695863746959, 'loss': 3.9888040555364603}


0